# 13.11 拿到分數後，為什麼還要扣掉預期？


一次選卡拿1分，另一次拿0分。若這種情境平常預期0.4分，兩次相對預期分別是0.6與−0.4。希望選擇規則更常作出比預期好的選擇，較少作出比預期差的選擇，才把評分轉成更新方向。

選卡的機率規則叫**策略（policy）**；估計作答前平均會拿多少分叫**基準值（baseline）**。實際分數減更新前的基準，叫**優勢估計（advantage）**。正負說的是高於或低於預期，不直接等於答案正誤。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.posttraining import bandit_advantage

rewards = torch.tensor([1.0, 0.0])
old_values = torch.tensor([0.4, 0.4], requires_grad=True)
advantage = bandit_advantage(rewards, old_values)
print("優勢估計", [round(value, 2) for value in advantage.tolist()])
print("更新策略時還追蹤基準梯度嗎", advantage.requires_grad)

`bandit_advantage`算`reward−old_value`，印出`[0.6,−0.4]`。最後`False`表示更新策略時，這份評語當成固定數值，不沿它的計算去改基準；基準模型另用自己的預測代價學習。

本章一次選整篇預寫卡，拿分便結束，是**依情境做一次選擇的任務（contextual bandit）**。所以優勢可用這個減法。逐token寫答案是一串選擇，早期文字會影響後文與最終回饋，需要另外處理各步回饋，不能把此例當成完整語言模型RLHF。

接下來的**近端策略最佳化**（Proximal Policy Optimization，PPO）是一種更新方法：它使用固定優勢，又看新舊選擇機率比，處理同一批舊作答被重用時的變化。reward尺度也須記下，真評分網路不必每次只有0與1。

練習把兩個舊基準改0.9，優勢變`[0.1,−0.9]`。分數沒有改，超出預期的程度改了。
